# CPU vs GPU Performance Analysis with CUDA (Google Colab)
This notebook benchmarks a simple vector addition workload on CPU and GPU using Numba CUDA.
It is designed for Google Colab. Enable a GPU runtime via Runtime > Change runtime type > GPU.

## Scope
- CPU vectorized NumPy baseline
- CPU parallel Numba (optional)
- GPU Numba CUDA kernel
- GPU end-to-end time includes host-device transfer

In [ ]:
import sys
import platform
import subprocess
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception as exc:
        return f"Command failed: {exc}"

In [ ]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Processor:", platform.processor() or "unknown")
print("nvidia-smi -L:")
print(run_cmd("nvidia-smi -L"))

## Install and verify Numba
Colab usually includes CUDA drivers, but Numba may not be installed in every runtime.

In [ ]:
import importlib.util

if importlib.util.find_spec("numba") is None:
    print("Installing numba...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])

from numba import cuda, njit, prange
import numba

print("Numba version:", numba.__version__)
print("CUDA available:", cuda.is_available())
if cuda.is_available():
    device = cuda.get_current_device()
    name = device.name.decode() if isinstance(device.name, bytes) else device.name
    print("CUDA device:", name)

## Kernels and benchmark helpers

In [ ]:
@cuda.jit
def vec_add_cuda(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

@njit(parallel=True, fastmath=True)
def vec_add_numba(a, b, out):
    for i in prange(a.size):
        out[i] = a[i] + b[i]

def time_numpy(a, b, out, repeats=10):
    times = []
    for _ in range(repeats):
        t0 = perf_counter()
        np.add(a, b, out=out)
        times.append(perf_counter() - t0)
    return min(times), float(np.median(times))

def time_numba(a, b, out, repeats=10):
    vec_add_numba(a, b, out)
    times = []
    for _ in range(repeats):
        t0 = perf_counter()
        vec_add_numba(a, b, out)
        times.append(perf_counter() - t0)
    return min(times), float(np.median(times))

def time_cuda(a, b, repeats=10, threads=256):
    n = a.size
    blocks = (n + threads - 1) // threads

    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_out = cuda.device_array_like(a)

    vec_add_cuda[blocks, threads](d_a, d_b, d_out)
    cuda.synchronize()

    kernel_ms = []
    for _ in range(repeats):
        start = cuda.event()
        end = cuda.event()
        start.record()
        vec_add_cuda[blocks, threads](d_a, d_b, d_out)
        end.record()
        end.synchronize()
        kernel_ms.append(start.elapsed_time(end))
    kernel_min = min(kernel_ms) / 1000.0
    kernel_med = float(np.median(kernel_ms)) / 1000.0

    e2e = []
    for _ in range(repeats):
        t0 = perf_counter()
        d_a = cuda.to_device(a)
        d_b = cuda.to_device(b)
        d_out = cuda.device_array_like(a)
        vec_add_cuda[blocks, threads](d_a, d_b, d_out)
        d_out.copy_to_host()
        cuda.synchronize()
        e2e.append(perf_counter() - t0)

    return (kernel_min, kernel_med), (min(e2e), float(np.median(e2e)))

## Benchmark configuration

In [ ]:
N = 10_000_000
REPEATS = 10
DTYPE = np.float32

rng = np.random.default_rng(0)
a = rng.random(N).astype(DTYPE)
b = rng.random(N).astype(DTYPE)
out = np.empty_like(a)

bytes_per_element = np.dtype(DTYPE).itemsize * 3
print("Array size:", N)
print("Array dtype:", DTYPE)

In [ ]:
results = []

cpu_min, cpu_med = time_numpy(a, b, out, repeats=REPEATS)
results.append({"method": "CPU NumPy", "min_s": cpu_min, "median_s": cpu_med})

numba_min, numba_med = time_numba(a, b, out, repeats=REPEATS)
results.append({"method": "CPU Numba parallel", "min_s": numba_min, "median_s": numba_med})

if cuda.is_available():
    (gpu_k_min, gpu_k_med), (gpu_e2e_min, gpu_e2e_med) = time_cuda(a, b, repeats=REPEATS)
    results.append({"method": "GPU kernel only", "min_s": gpu_k_min, "median_s": gpu_k_med})
    results.append({"method": "GPU end-to-end", "min_s": gpu_e2e_min, "median_s": gpu_e2e_med})
else:
    print("CUDA not available. GPU benchmarks skipped.")

if cuda.is_available():
    threads = 256
    blocks = (N + threads - 1) // threads
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_out = cuda.device_array_like(a)
    vec_add_cuda[blocks, threads](d_a, d_b, d_out)
    gpu_out = d_out.copy_to_host()
    max_err = float(np.max(np.abs(gpu_out - (a + b))))
    print("Max error:", max_err)

try:
    import pandas as pd
    from IPython.display import display
    df = pd.DataFrame(results)
    df["throughput_GB_s"] = (N * bytes_per_element / 1e9) / df["median_s"]
    base = df.loc[df["method"] == "CPU NumPy", "median_s"].iloc[0]
    df["speedup_vs_numpy"] = base / df["median_s"]
    display(df)
except Exception as exc:
    print("Pandas not available:", exc)
    for row in results:
        print(row)

In [ ]:
labels = [r["method"] for r in results]
medians = [r["median_s"] for r in results]

plt.figure(figsize=(8, 4))
colors = ["#3b82f6", "#10b981", "#f59e0b", "#ef4444"]
plt.bar(labels, medians, color=colors[: len(labels)])
plt.ylabel("Median time (s)")
plt.title("CPU vs GPU Vector Add")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Interactive Gradio UI
Run a web UI to benchmark with custom settings. The UI launches a public link when `share=True`.


In [ ]:
import importlib.util

if importlib.util.find_spec("gradio") is None:
    print("Installing gradio...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])

if importlib.util.find_spec("pandas") is None:
    print("Installing pandas...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import gradio as gr
import pandas as pd


def run_benchmark_ui(n, repeats, dtype_name, use_gpu, threads, seed):
    n = int(n)
    repeats = int(repeats)
    threads = int(threads)
    seed = int(seed)

    if n <= 0:
        return pd.DataFrame(), None, "N must be positive."

    dtype_map = {"float32": np.float32, "float64": np.float64}
    dtype = dtype_map[dtype_name]

    rng = np.random.default_rng(seed)
    a = rng.random(n).astype(dtype)
    b = rng.random(n).astype(dtype)
    out = np.empty_like(a)

    bytes_per_element = np.dtype(dtype).itemsize * 3
    results = []

    cpu_min, cpu_med = time_numpy(a, b, out, repeats=repeats)
    results.append({"method": "CPU NumPy", "min_s": cpu_min, "median_s": cpu_med})

    numba_min, numba_med = time_numba(a, b, out, repeats=repeats)
    results.append({"method": "CPU Numba parallel", "min_s": numba_min, "median_s": numba_med})

    info_lines = [
        f"N: {n}",
        f"dtype: {dtype_name}",
        f"repeats: {repeats}",
    ]

    if use_gpu:
        if cuda.is_available():
            (gpu_k_min, gpu_k_med), (gpu_e2e_min, gpu_e2e_med) = time_cuda(
                a, b, repeats=repeats, threads=threads
            )
            results.append({"method": "GPU kernel only", "min_s": gpu_k_min, "median_s": gpu_k_med})
            results.append({"method": "GPU end-to-end", "min_s": gpu_e2e_min, "median_s": gpu_e2e_med})
            info_lines.append("CUDA: available")
            info_lines.append(f"threads per block: {threads}")

            blocks = (n + threads - 1) // threads
            d_a = cuda.to_device(a)
            d_b = cuda.to_device(b)
            d_out = cuda.device_array_like(a)
            vec_add_cuda[blocks, threads](d_a, d_b, d_out)
            gpu_out = d_out.copy_to_host()
            max_err = float(np.max(np.abs(gpu_out - (a + b))))
            info_lines.append(f"max error: {max_err:.3e}")
        else:
            info_lines.append("CUDA: not available - GPU skipped")

    df = pd.DataFrame(results)
    if not df.empty:
        df["throughput_GB_s"] = (n * bytes_per_element / 1e9) / df["median_s"]
        base = df.loc[df["method"] == "CPU NumPy", "median_s"].iloc[0]
        df["speedup_vs_numpy"] = base / df["median_s"]

    fig = None
    if not df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#3b82f6", "#10b981", "#f59e0b", "#ef4444"]
        ax.bar(df["method"], df["median_s"], color=colors[: len(df)])
        ax.set_ylabel("Median time (s)")
        ax.set_title("CPU vs GPU Vector Add")
        ax.tick_params(axis="x", labelrotation=20)
        fig.tight_layout()

    return df, fig, "\n".join(info_lines)


with gr.Blocks(title="CPU vs GPU Performance (CUDA)") as demo:
    gr.Markdown("# CPU vs GPU Performance (CUDA)")
    gr.Markdown("Run this on Colab with a GPU runtime. A public link is created with `share=True`.")

    with gr.Row():
        with gr.Column(scale=1):
            n_in = gr.Number(value=10_000_000, precision=0, label="Vector length (N)")
            repeats_in = gr.Slider(3, 20, value=10, step=1, label="Repeats")
            dtype_in = gr.Dropdown(["float32", "float64"], value="float32", label="Data type")
            use_gpu_in = gr.Checkbox(value=True, label="Use GPU (if available)")
            threads_in = gr.Slider(64, 1024, value=256, step=32, label="CUDA threads per block")
            seed_in = gr.Number(value=0, precision=0, label="RNG seed")
            run_btn = gr.Button("Run benchmark", variant="primary")
        with gr.Column(scale=2):
            df_out = gr.Dataframe(label="Results", interactive=False)
            plot_out = gr.Plot(label="Timing chart")
            info_out = gr.Textbox(label="Run info", lines=6)

    run_btn.click(
        run_benchmark_ui,
        inputs=[n_in, repeats_in, dtype_in, use_gpu_in, threads_in, seed_in],
        outputs=[df_out, plot_out, info_out],
        api_name="benchmark",
    )

demo.launch(share=True)


## Notes
- GPU kernel-only time excludes host-device transfer. End-to-end time is more realistic for small workloads.
- For larger or more compute-heavy workloads, the GPU speedup is typically higher.
- If you change N, re-run the benchmark cells to refresh the results.